In [17]:
# DASK client set

import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

# get path for path changes in Jupyter notebook: File - Open from Path - insert relative_path
notebook_path = os.path.abspath(".")
_, _, relative_path = notebook_path.partition('/all/')
relative_path = '/all/' + relative_path
relative_path

'/all/Model/CESM2/Earth_System_Predictability/DIC'

# Load modules

In [18]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.ticker as mticker
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.interpolate import griddata
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp


In [19]:
import xcesm
import gsw

In [20]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_NWP_preprocessing import CESM2_NWP_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

# Variable configuration

In [21]:

cfgs = {}
# varlist = ["DIC", "TEMP", "SALT"]
# varlist = ["DIC_ALT_CO2"]
varlist = ["FG_CO2"]

for varname in varlist:
    cfg = CESM2_NWP_config()
    cfg.year_s = 1955
    cfg.year_e = 2020
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')

# Read dataset

In [22]:
# lat_range = slice(23, 38)
# lon_range = slice(120, 180)
lat_range = slice(19, 40)
lon_range = slice(120, 180)

In [23]:
# define preprocessing function

# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]
# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'dz', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]


# exceptcv = [
#     'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
# ] + [cfg.var for cfg in cfgs.values()]
exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 
] + [cfg.var for cfg in cfgs.values()]


def process_coords_3d(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        # ds = ds.isel(z_t=slice(0, 39)) # ~39 layer (1000m)
        # ds = (ds.isel(z_t=slice(1, 39)) * ds.dz).sum(dim='z_t') / ds.dz.sum(dim='z_t')
        return ds
    else:
        return ds.set_coords(coord_vars)



def process_coords_3d_LE(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """
    Preprocessor function for CESM POP-style datasets.
    - Normalizes vertical coordinate: if z_t or z_t_2 exists, rename to 'depth'.
    - Replaces its values with z_t_new for consistency.
    - Optionally drops unnecessary coordinate variables for faster concatenation.
    """
    z_t_new = np.array([5.0000000e+00, 1.5000000e+01, 2.5000000e+01, 3.5000000e+01,
       4.5000000e+01, 5.5000000e+01, 6.5000000e+01, 7.5000000e+01,
       8.5000000e+01, 9.5000000e+01, 1.0500000e+02, 1.1500000e+02,
       1.2500000e+02, 1.3500000e+02, 1.4500000e+02, 1.5500000e+02,
       1.6509839e+02, 1.7547903e+02, 1.8629126e+02, 1.9766026e+02,
       2.0971138e+02, 2.2257828e+02, 2.3640883e+02, 2.5137015e+02,
       2.6765421e+02, 2.8548364e+02, 3.0511920e+02, 3.2686798e+02,
       3.5109348e+02, 3.7822760e+02, 4.0878464e+02, 4.4337769e+02,
       4.8273669e+02, 5.2772797e+02, 5.7937286e+02, 6.3886261e+02,
       7.0756329e+02, 7.8700250e+02, 8.7882520e+02, 9.8470581e+02,
       1.1062042e+03, 1.2445669e+03, 1.4004972e+03, 1.5739464e+03,
       1.7640033e+03, 1.9689442e+03, 2.1864565e+03, 2.4139714e+03,
       2.6490012e+03, 2.8893845e+03, 3.1334045e+03, 3.3797935e+03,
       3.6276702e+03, 3.8764519e+03, 4.1257681e+03, 4.3753926e+03,
       4.6251904e+03, 4.8750835e+03, 5.1250278e+03, 5.3750000e+03])
    
    # ------------------------------------------------------
    # 1️⃣ Normalize vertical coordinate name
    # ------------------------------------------------------
    if "z_t_2" in ds.dims:
        ds = ds.rename({"z_t_2": "depth"})
    elif "z_t" in ds.dims:
        ds = ds.rename({"z_t": "depth"})
    else:
        print("[Warning] No vertical coordinate (z_t or z_t_2) found — skipped.")
        return ds

    # Drop any leftover z_t/z_t_2 coordinate variable if it exists
    ds = ds.drop_vars(["z_t", "z_t_2"], errors="ignore")

    # ------------------------------------------------------
    # 2️⃣ Replace coordinate values with z_t_new
    # ------------------------------------------------------
    if "depth" in ds.coords:
        if len(ds["depth"]) == len(z_t_new):
            ds = ds.assign_coords(depth=z_t_new)
        else:
            print(f"[Warning] depth length mismatch: {len(ds['depth'])} vs {len(z_t_new)}")
    else:
        print("[Warning] depth coordinate missing after renaming.")

    # ------------------------------------------------------
    # 3️⃣ Clean up coordinate references inside variable attributes
    # ------------------------------------------------------
    for v in ds.data_vars:
        if "coordinates" in ds[v].attrs:
            ds[v].attrs["coordinates"] = (
                ds[v].attrs["coordinates"]
                .replace("z_t_2", "depth")
                .replace("z_t", "depth")
            )

    # ------------------------------------------------------
    # 4️⃣ Drop unnecessary coordinate variables and slice time
    # ------------------------------------------------------
    coord_vars = []
    for v in np.array(ds.coords):
        if v not in except_coord_vars:
            coord_vars.append(v)
    for v in np.array(ds.data_vars):
        if v not in except_coord_vars:
            coord_vars.append(v)

    if drop:
        ds = ds.drop(coord_vars, errors="ignore")
        ds = ds.sel(time=slice(sd, ed))
    else:
        ds = ds.set_coords(coord_vars)

    return ds



# exceptcv=['time','lon','lat','lev', 'TLONG', 'TLAT', 'z_t', cfg_var_DIC.var, cfg_var_Li_diat.var, cfg_var_Li_sp.var,
#          cfg_var_Vi_diat_Fe.var, cfg_var_Vi_diat_N.var, cfg_var_Vi_diat_P.var, cfg_var_Vi_diat_SiO3.var,
#          cfg_var_Vi_sp_Fe.var, cfg_var_Vi_sp_N.var, cfg_var_Vi_sp_P.var,
#          cfg_var_NPP_diat.var, cfg_var_NPP_sp.var]
# def process_coords(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
#     """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
#     coord_vars = []
#     for v in np.array(ds.coords) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     for v in np.array(ds.data_vars) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
    
#     if drop:
#         ds= ds.drop(coord_vars)
#         ds= ds.sel(time=slice(sd, ed))
#         return ds
#     else:
#         return ds.set_coords(coord_vars)

start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)


# ds = ds.isel(lev=slice(1, 11))

In [24]:
vars = varlist
results = {}

for var in vars:
    print(f"\n===== Processing {var} =====")

    cfg_var = cfgs[var]
    cfg_var.var = var

    # ================= ODA =================
    t0 = time.time()
    cfg_var.ODA_path_load(cfg_var.var)
    cfg_var.ODA_ds = xr.open_mfdataset(
        cfg_var.ODA_file_list[0][15:16],
        chunks={'time': 12},
        combine='nested',
        concat_dim=[[*cfg_var.ODA_ensembles][15:16], 'time'],
        parallel=True,
        preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
        decode_cf=True,
        decode_times=True,
    )
    cfg_var.ODA_ds = cfg_var.ODA_ds.rename({"concat_dim": "ens_ODA"})
    new_time = cfg_var.ODA_ds.time - np.array(
        [datetime.timedelta(days=15)] * len(cfg_var.ODA_ds.time)
    )
    cfg_var.ODA_ds = cfg_var.ODA_ds.assign_coords(time=new_time)
    print(f"  ODA read: {time.time() - t0:.1f} s")

    

    # ================= REGRID =================
    t0 = time.time()

    cfg_var.ODA_ds_rgd = (
        cfg_var.ODA_ds[var]
        .isel(ens_ODA=0)
        .utils.regrid()
        .sel(lat=lat_range, lon=lon_range)
        .sortby("time")
    )
    print(f"  REGRID ODA: {time.time() - t0:.1f} s")
   
    # cfg_var.LE_ds_rgd = cfg_var.LE_ds_rgd.assign_coords(
    #     z_t=cfg_var.ODA_ds_rgd.z_t
    # )
    

    results[var] = cfg_var



===== Processing FG_CO2 =====
  ODA read: 2.7 s
  REGRID ODA: 1.2 s


In [25]:
import xarray as xr
import numpy as np
import datetime
import time
import os

OUT_BASE = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/LENS2"

vars = varlist
results = {}

for var in vars:
    print(f"\n===== Processing {var} =====")

    cfg_var = cfgs[var]
    cfg_var.var = var

    # Load file paths
    cfg_var.LE_path_load(var)

    for ens_idx, ens_name in enumerate(cfg_var.LE_ensembles):
        print(f"\n--- LE ensemble {ens_name} ---")

        file_list = cfg_var.LE_file_list[0][ens_idx]

        for i, in_file in enumerate(file_list):
            print(f"[{ens_name}] {i+1}/{len(file_list)}")
            print(f"INPUT : {in_file}")

            t0 = time.time()

            # Open single file
            ds = xr.open_dataset(
                in_file,
                chunks={"time": 12},
                decode_cf=True,
                decode_times=True,
            )

            ds = process_coords_3d_LE(
                ds, start_date, end_date
            )

            # Time shift
            new_time = ds.time - np.array(
                [datetime.timedelta(days=15)] * len(ds.time)
            )
            ds = ds.assign_coords(time=new_time)

            # Rename depth dimension if needed
            if "depth" in ds.dims:
                ds = ds.rename({"depth": "z_t"})

            # Regrid
            da_rgd = (
                ds[var]
                .utils.regrid()
                .sel(lat=lat_range, lon=lon_range)
                .sortby("time")
            )

            # Extract path components
            parts = in_file.split("/")
            comp = parts[-5]       # ocn
            varname = parts[-4]    # DIC
            ens_dir = parts[-3]    # b.e21...
            subdir = parts[-2]     # NWP
            filename = parts[-1]   # NWP_...nc

            # Build new directory structure
            new_dir = os.path.join(
                OUT_BASE,
                subdir,    # NWP first
                comp,      # ocn
                varname,   # DIC
                ens_dir,   # ensemble directory
            )
            os.makedirs(new_dir, exist_ok=True)

            # Add regridded_ prefix to filename
            new_filename = f"regridded_{filename}"
            out_file = os.path.join(new_dir, new_filename)

            # Save
            encoding = {
                var: {
                    "zlib": True,
                    "complevel": 4,
                    "dtype": "float32",
                }
            }

            da_rgd.to_netcdf(out_file, encoding=encoding)

            ds.close()
            del ds, da_rgd

            print(f"OUTPUT: {out_file}")
            print(f"Saved in {time.time() - t0:.1f} s\n")

    results[var] = cfg_var



===== Processing FG_CO2 =====

--- LE ensemble 0 ---
[0] 1/8
INPUT : /mnt/lustre/proj/earth.system.predictability/LENS2/archive_region_transfer/ocn/FG_CO2/b.e21.BHISTsmbb.f09_g17.LE2-1011.001/NWP/NWP_b.e21.BHISTsmbb.f09_g17.LE2-1011.001.pop.h.FG_CO2.195001-195912.nc
[Warning] depth length mismatch: 1 vs 60
OUTPUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/LENS2/NWP/ocn/FG_CO2/b.e21.BHISTsmbb.f09_g17.LE2-1011.001/regridded_NWP_b.e21.BHISTsmbb.f09_g17.LE2-1011.001.pop.h.FG_CO2.195001-195912.nc
Saved in 0.3 s

[0] 2/8
INPUT : /mnt/lustre/proj/earth.system.predictability/LENS2/archive_region_transfer/ocn/FG_CO2/b.e21.BHISTsmbb.f09_g17.LE2-1011.001/NWP/NWP_b.e21.BHISTsmbb.f09_g17.LE2-1011.001.pop.h.FG_CO2.196001-196912.nc
[Warning] depth length mismatch: 1 vs 60
OUTPUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/LENS2/NWP/ocn/FG_CO2/b.e21.BHISTsmbb.f09_g17.LE2-1011.001/regridded_NWP_b.e21.BHISTsmbb.f09_g17.LE2-1011.001.pop.h.FG_CO2.196001-196912.nc
Saved in 0.4 s

[0] 3/8
INPUT : 